# 🤝 A2A (Agent-to-Agent) マルチエージェント with Phi-3.5

各エージェントがHTTPサーバーとして起動し、**A2Aプロトコル**でメッセージを送り合います。

```
ユーザーのテーマ
       ↓
┌─────────────────────────────────────────┐
│  🔍 調査エージェント (localhost:5001)    │
│  A2AServer として起動                   │
└───────────────┬─────────────────────────┘
                ↓ HTTP POST (A2Aプロトコル)
┌─────────────────────────────────────────┐
│  📊 分析エージェント (localhost:5002)    │
│  調査結果を受け取りA2Aメッセージで分析  │
│  → 追加調査依頼もA2Aで送信             │
└───────────────┬─────────────────────────┘
                ↓ HTTP POST (A2Aプロトコル)
┌─────────────────────────────────────────┐
│  ✍️  執筆エージェント (localhost:5003)   │
│  全結果を統合して最終回答を執筆         │
└─────────────────────────────────────────┘
```

## 前バージョン（Pythonオブジェクト直接呼び出し）との違い

| | 前バージョン | 本ノートブック（A2A） |
|---|---|---|
| 通信方式 | Pythonの戻り値（関数呼び出し） | HTTP POST（A2Aプロトコル） |
| エージェントの独立性 | 同一プロセス内 | 各エージェントが独立したHTTPサーバー |
| メッセージ形式 | 文字列・辞書 | A2A標準メッセージ（Message/TextContent） |
| 異なるフレームワーク間連携 | 不可 | 可能（A2A互換なら何でもOK） |
| AgentCard（自己紹介） | なし | あり（名前・説明・スキルを公開） |

## Step 1: インストール

In [ ]:
# llama-cpp-python（LLM）と python-a2a（A2Aプロトコル）をインストール
import subprocess, os

subprocess.run(
    ['pip', 'install', 'llama-cpp-python',
     '--extra-index-url', 'https://abetlen.github.io/llama-cpp-python/whl/cu121',
     '--upgrade', '-q'],
    env={**os.environ, 'CMAKE_ARGS': '-DGGML_CUDA=on'},
    capture_output=True,
)
subprocess.run(['pip', 'install', 'python-a2a', '-q'], capture_output=True)

print('✅ インストール完了 → ランタイムを再起動してください')

⚠️ **ランタイムを再起動してから Step 2 以降を実行してください**

## Step 2: GPU確認 & モデルダウンロード

In [ ]:
import torch
from llama_cpp import Llama
from python_a2a import A2AServer, A2AClient, AgentCard, Message, TextContent, MessageRole, run_server

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✅ GPU: {gpu} ({vram:.1f} GB VRAM)')
else:
    print('⚠️  GPU未検出')
print('✅ llama-cpp-python & python-a2a OK')

In [ ]:
%%bash
MODEL_PATH="/content/phi-3.5-mini-q4.gguf"
MODEL_URL="https://huggingface.co/bartowski/Phi-3.5-mini-instruct-GGUF/resolve/main/Phi-3.5-mini-instruct-Q4_K_M.gguf"
if [ -f "$MODEL_PATH" ]; then
  echo "✅ モデルは既にダウンロード済みです"
else
  echo "📥 ダウンロード中... (~2.2GB)"
  wget -q --show-progress -O "$MODEL_PATH" "$MODEL_URL"
  echo "✅ ダウンロード完了"
fi
ls -lh "$MODEL_PATH"

## Step 3: モデルロード & 共通関数

In [ ]:
from llama_cpp import Llama
import re, math, threading, time

print('📥 Phi-3.5 をロード中...')
llm = Llama(
    model_path='/content/phi-3.5-mini-q4.gguf',
    n_gpu_layers=35,
    n_ctx=4096,
    n_batch=512,
    verbose=False,
    chat_format='chatml',
)
print('✅ モデルロード完了')

def llm_chat(messages: list, max_tokens: int = 512, temperature: float = 0.5) -> str:
    resp = llm.create_chat_completion(
        messages=messages,
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=0.9,
        repeat_penalty=1.1,
        stop=['<|end|>', '<|user|>'],
    )
    return resp['choices'][0]['message']['content'].strip()

def search_knowledge(query: str) -> str:
    kb = {
        'python':    'Pythonは1991年にGuido van Rossumが開発した高水準言語。読みやすい文法と豊富なライブラリが特徴。',
        '機械学習':  '機械学習はデータからパターンを自動的に学習するAI技術。教師あり・教師なし・強化学習の3種がある。',
        'phi':       'Phi-3.5はMicrosoftが開発した3.8Bパラメータの小型高性能LLM。MITライセンスで登録不要・商用利用可能。',
        'llama':     'llama-cpp-pythonはC++製LLM推論エンジンllama.cppのPythonバインディング。pip installだけで使えてGPU加速対応。',
        '深層学習':  '深層学習は多層ニューラルネットワークを用いる機械学習の手法。画像認識・自然言語処理で革新的な性能を達成した。',
        'a2a':       'A2A (Agent-to-Agent) はGoogleが2025年4月に発表したエージェント間通信プロトコル。JSON-RPC 2.0 over HTTPで標準化されており、150以上の組織が対応。',
        '東京':      '東京は日本の首都。人口約1,400万人（都区部）。世界最大級の都市圏の一つ。',
    }
    for key, val in kb.items():
        if key.lower() in query.lower():
            return val
    return f'「{query}」に関する情報は見つかりませんでした。'

print('✅ 共通関数定義完了')

## Step 4: A2Aエージェントの定義

各エージェントは `A2AServer` を継承し、`AgentCard`（自己紹介カード）を持ちます。  
エージェント間の通信は `A2AClient` が HTTP POST で行います。

In [ ]:
from python_a2a import A2AServer, A2AClient, AgentCard, Message, TextContent, MessageRole, run_server

# ============================================================
# 🔍 調査エージェント (port: 5001)
# ============================================================
class ResearchAgent(A2AServer):
    def __init__(self):
        agent_card = AgentCard(
            name='ResearchAgent',
            description='テーマに関する情報を収集・整理する調査専門エージェント',
            url='http://localhost:5001/a2a',
            version='1.0.0',
        )
        super().__init__(agent_card=agent_card)

    def handle_message(self, message: Message) -> Message:
        """A2Aメッセージを受け取り、調査結果をA2Aメッセージで返す"""
        query = message.content.text
        print(f'  [🔍 調査] 受信: {query[:60]}...' if len(query) > 60 else f'  [🔍 調査] 受信: {query}')

        # ナレッジベース検索
        kb_result = search_knowledge(query)

        # LLMで調査レポートを生成
        result = llm_chat([
            {'role': 'system', 'content': '調査専門AIです。与えられた情報をもとに、事実に基づいた調査レポートを日本語で簡潔に作成してください。'},
            {'role': 'user',   'content': f'テーマ: {query}\n\nナレッジベース: {kb_result}'},
        ], max_tokens=300)

        print(f'  [🔍 調査] 送信: {result[:60]}...' if len(result) > 60 else f'  [🔍 調査] 送信: {result}')

        # A2Aメッセージとして返す
        return Message(
            content=TextContent(text=result),
            role=MessageRole.AGENT,
            parent_message_id=message.message_id,
            conversation_id=message.conversation_id,
        )


# ============================================================
# 📊 分析エージェント (port: 5002)
# ============================================================
class AnalysisAgent(A2AServer):
    def __init__(self):
        agent_card = AgentCard(
            name='AnalysisAgent',
            description='調査結果を多角的に分析・評価する分析専門エージェント',
            url='http://localhost:5002/a2a',
            version='1.0.0',
        )
        super().__init__(agent_card=agent_card)
        # 調査エージェントへのA2Aクライアント
        self.research_client = A2AClient('http://localhost:5001/a2a')

    def handle_message(self, message: Message) -> Message:
        """A2Aメッセージを受け取り、必要に応じて調査エージェントにA2Aで追加調査を依頼する"""
        text = message.content.text
        print(f'  [📊 分析] 受信: {text[:60]}...' if len(text) > 60 else f'  [📊 分析] 受信: {text}')

        # LLMで分析（追加調査依頼も生成）
        analysis = llm_chat([
            {'role': 'system', 'content': '分析専門AIです。調査結果を批判的・多角的に分析してください。'
                                          '情報が不足していれば「追加調査依頼:」に続けて依頼内容を書いてください。'
                                          '日本語で回答してください。'},
            {'role': 'user',   'content': f'調査結果:\n{text}'},
        ], max_tokens=400)

        # 追加調査依頼があれば調査エージェントにA2Aで送信
        if '追加調査依頼:' in analysis:
            parts = analysis.split('追加調査依頼:')
            additional_req = parts[1].strip() if len(parts) > 1 else ''
            analysis = parts[0].strip()

            if additional_req:
                print(f'  [📊 分析] → 調査エージェントに追加調査をA2Aで送信: {additional_req[:50]}')
                # A2Aプロトコルで調査エージェントに追加調査依頼
                extra_msg = Message(
                    content=TextContent(text=additional_req),
                    role=MessageRole.USER,
                )
                extra_result = self.research_client.send_message(extra_msg)
                analysis += f'\n\n【追加調査結果】\n{extra_result.content.text}'

        print(f'  [📊 分析] 送信: {analysis[:60]}...' if len(analysis) > 60 else f'  [📊 分析] 送信: {analysis}')

        return Message(
            content=TextContent(text=analysis),
            role=MessageRole.AGENT,
            parent_message_id=message.message_id,
            conversation_id=message.conversation_id,
        )


# ============================================================
# ✍️  執筆エージェント (port: 5003)
# ============================================================
class WritingAgent(A2AServer):
    def __init__(self):
        agent_card = AgentCard(
            name='WritingAgent',
            description='調査・分析結果を統合して最終回答を執筆する執筆専門エージェント',
            url='http://localhost:5003/a2a',
            version='1.0.0',
        )
        super().__init__(agent_card=agent_card)

    def handle_message(self, message: Message) -> Message:
        """調査・分析の統合テキストを受け取り、最終回答をA2Aメッセージで返す"""
        text = message.content.text
        print(f'  [✍️  執筆] 受信: {text[:60]}...' if len(text) > 60 else f'  [✍️  執筆] 受信: {text}')

        result = llm_chat([
            {'role': 'system', 'content': '執筆専門AIです。調査・分析の内容を統合し、読みやすい最終回答を日本語で作成してください。'},
            {'role': 'user',   'content': text},
        ], max_tokens=512)

        print(f'  [✍️  執筆] 送信: {result[:60]}...' if len(result) > 60 else f'  [✍️  執筆] 送信: {result}')

        return Message(
            content=TextContent(text=result),
            role=MessageRole.AGENT,
            parent_message_id=message.message_id,
            conversation_id=message.conversation_id,
        )


print('✅ 3つのA2Aエージェントクラス定義完了')
print('  🔍 ResearchAgent  → localhost:5001')
print('  📊 AnalysisAgent  → localhost:5002')
print('  ✍️  WritingAgent   → localhost:5003')

## Step 5: A2Aサーバーを起動

各エージェントをスレッドで並列起動します。  
起動後は `http://localhost:500X/a2a` にHTTP POSTするだけでどのクライアントからも呼び出せます。

In [ ]:
import threading, time, requests

def start_agent(agent_instance, port: int):
    """エージェントをバックグラウンドスレッドで起動"""
    t = threading.Thread(
        target=run_server,
        kwargs={'agent': agent_instance, 'host': '127.0.0.1', 'port': port},
        daemon=True,
    )
    t.start()
    return t

def wait_for_server(port: int, name: str, timeout: int = 30):
    """サーバーの起動を待機"""
    for _ in range(timeout):
        try:
            requests.get(f'http://127.0.0.1:{port}/a2a', timeout=1)
            print(f'  ✅ {name} 起動完了 (localhost:{port})')
            return True
        except:
            time.sleep(1)
    print(f'  ❌ {name} 起動失敗')
    return False

print('🚀 A2Aサーバー起動中...')

start_agent(ResearchAgent(), port=5001)
start_agent(AnalysisAgent(), port=5002)
start_agent(WritingAgent(),  port=5003)

time.sleep(2)
wait_for_server(5001, '🔍 ResearchAgent')
wait_for_server(5002, '📊 AnalysisAgent')
wait_for_server(5003, '✍️  WritingAgent')

print('\n✅ 全エージェント起動完了')

## Step 6: AgentCardを確認

A2Aプロトコルでは各エージェントが `AgentCard`（自己紹介カード）を公開します。  
クライアントはこれを見てエージェントの能力を発見できます。

In [ ]:
import requests, json

print('📋 AgentCard（各エージェントの自己紹介）')
print('=' * 55)

for port, emoji in [(5001, '🔍'), (5002, '📊'), (5003, '✍️ ')]:
    try:
        # A2Aプロトコル: GET /.well-known/agent.json でAgentCardを取得
        r = requests.get(f'http://localhost:{port}/agent.json', timeout=3)
        card = r.json()
        print(f'\n{emoji} Port {port}')
        print(f'  名前: {card.get("name", "N/A")}')
        print(f'  説明: {card.get("description", "N/A")}')
        print(f'  URL:  {card.get("url", "N/A")}')
        print(f'  Ver:  {card.get("version", "N/A")}')
    except Exception as e:
        print(f'\n{emoji} Port {port}: {e}')

## Step 7: オーケストレーター（A2Aクライアント）

In [ ]:
from python_a2a import A2AClient, Message, TextContent, MessageRole

# 各エージェントへのA2Aクライアント
research_client = A2AClient('http://localhost:5001/a2a')
analysis_client = A2AClient('http://localhost:5002/a2a')
writing_client  = A2AClient('http://localhost:5003/a2a')


def run_a2a_multi_agent(topic: str) -> dict:
    """
    A2Aプロトコルで3エージェントを連携させてタスクを処理する

    通信フロー:
      1. オーケストレーター → 調査エージェント (A2A HTTP POST)
      2. オーケストレーター → 分析エージェント (A2A HTTP POST)
         └─ 分析エージェント → 調査エージェント (A2A HTTP POST / 必要な場合)
      3. オーケストレーター → 執筆エージェント (A2A HTTP POST)
    """
    SEP  = '─' * 55
    SEP2 = '┄' * 55
    log  = []

    print(f'\n{SEP}')
    print(f'🤝 A2Aマルチエージェント起動')
    print(f'📋 テーマ: {topic}')
    print(SEP)

    # ──────────────────────────────────────────────
    # Phase 1: 調査エージェントにA2Aメッセージを送信
    # ──────────────────────────────────────────────
    print(f'\n{SEP2}')
    print('📤 [オーケストレーター → 🔍 調査エージェント] A2Aメッセージ送信')
    research_msg = Message(
        content=TextContent(text=topic),
        role=MessageRole.USER,
    )
    research_resp = research_client.send_message(research_msg)
    research_result = research_resp.content.text
    print(f'📥 [🔍 調査エージェント → オーケストレーター] A2Aレスポンス受信')
    print(f'   {research_result[:120]}...' if len(research_result) > 120 else f'   {research_result}')
    log.append({'from': 'orchestrator', 'to': 'research', 'content': topic})
    log.append({'from': 'research', 'to': 'orchestrator', 'content': research_result})

    # ──────────────────────────────────────────────
    # Phase 2: 分析エージェントにA2Aメッセージを送信
    #          （分析エージェントは内部で調査エージェントに追加依頼する場合あり）
    # ──────────────────────────────────────────────
    print(f'\n{SEP2}')
    print('📤 [オーケストレーター → 📊 分析エージェント] A2Aメッセージ送信')
    analysis_msg = Message(
        content=TextContent(text=f'テーマ: {topic}\n\n調査結果:\n{research_result}'),
        role=MessageRole.USER,
    )
    analysis_resp = analysis_client.send_message(analysis_msg)
    analysis_result = analysis_resp.content.text
    print(f'📥 [📊 分析エージェント → オーケストレーター] A2Aレスポンス受信')
    print(f'   {analysis_result[:120]}...' if len(analysis_result) > 120 else f'   {analysis_result}')
    log.append({'from': 'orchestrator', 'to': 'analysis', 'content': f'テーマ:{topic} / 調査結果:{research_result[:50]}...'})
    log.append({'from': 'analysis', 'to': 'orchestrator', 'content': analysis_result})

    # ──────────────────────────────────────────────
    # Phase 3: 執筆エージェントにA2Aメッセージを送信
    # ──────────────────────────────────────────────
    print(f'\n{SEP2}')
    print('📤 [オーケストレーター → ✍️  執筆エージェント] A2Aメッセージ送信')
    writing_msg = Message(
        content=TextContent(
            text=f'テーマ: {topic}\n\n【調査結果】\n{research_result}\n\n【分析結果】\n{analysis_result}'
        ),
        role=MessageRole.USER,
    )
    writing_resp = writing_client.send_message(writing_msg)
    final_answer = writing_resp.content.text
    print(f'📥 [✍️  執筆エージェント → オーケストレーター] A2Aレスポンス受信')
    log.append({'from': 'orchestrator', 'to': 'writing', 'content': '調査+分析結果を送信'})
    log.append({'from': 'writing', 'to': 'orchestrator', 'content': final_answer})

    print(f'\n{SEP}')
    print('✅ 最終回答')
    print(SEP)
    print(final_answer)
    print(SEP)

    return {'topic': topic, 'log': log, 'final_answer': final_answer}


print('✅ オーケストレーター定義完了')

## Step 8: 実行してみよう！

In [ ]:
# 例1: 機械学習について
result = run_a2a_multi_agent('機械学習')

In [ ]:
# 例2: A2Aプロトコルについて
result = run_a2a_multi_agent('A2Aプロトコル')

In [ ]:
# 例3: 自由入力
MY_TOPIC = '深層学習'  # ← ここを変えてください
result = run_a2a_multi_agent(MY_TOPIC)

## Step 9: A2A通信ログを確認

In [ ]:
print('\n📜 A2A通信ログ（エージェント間のメッセージ履歴）')
print('=' * 55)
for i, entry in enumerate(result['log'], 1):
    print(f"\n[{i}] {entry['from']} → {entry['to']}")
    content = entry['content']
    print(content[:200] + '...' if len(content) > 200 else content)
    print('┄' * 55)

## Step 10: インタラクティブUI

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

topic_box = widgets.Text(
    value='機械学習',
    description='テーマ:',
    layout=widgets.Layout(width='60%'),
)
run_btn = widgets.Button(
    description='🤝 A2Aエージェント実行',
    button_style='primary',
    layout=widgets.Layout(width='240px', height='38px'),
)
out = widgets.Output()

def on_click(b):
    with out:
        clear_output()
        run_a2a_multi_agent(topic_box.value)

run_btn.on_click(on_click)
display(topic_box, run_btn, out)

---
## 📚 カスタマイズガイド

### エージェントを追加する
```python
class ReviewAgent(A2AServer):
    def __init__(self):
        agent_card = AgentCard(
            name='ReviewAgent',
            description='最終回答をレビューする',
            url='http://localhost:5004/a2a',
            version='1.0.0',
        )
        super().__init__(agent_card=agent_card)

    def handle_message(self, message: Message) -> Message:
        result = llm_chat([...], max_tokens=300)
        return Message(
            content=TextContent(text=result),
            role=MessageRole.AGENT,
            parent_message_id=message.message_id,
            conversation_id=message.conversation_id,
        )

start_agent(ReviewAgent(), port=5004)
review_client = A2AClient('http://localhost:5004/a2a')
```

### 外部サービスのA2Aエージェントと連携する
```python
# A2A互換であれば、どんなエージェントとも接続できる
external_client = A2AClient('https://external-agent.example.com/a2a')
response = external_client.send_message(
    Message(content=TextContent(text='質問内容'), role=MessageRole.USER)
)
```